# MoMo Fraud Detection — Model Evaluation
**Step 5 of 6**

## 0. Install Dependencies

In [ ]:
import subprocess,sys
for pkg in ['pandas','numpy','scikit-learn','xgboost','matplotlib','seaborn']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
print('Ready')

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.metrics import (classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, f1_score)
sns.set_theme(style='whitegrid')

## 2. Load Test Data and Models

In [ ]:
df = pd.read_csv('Pay-sim_features.csv')
from sklearn.model_selection import train_test_split
X = df.drop(columns=['isFraud'])
y = df['isFraud']
_,X_test,_,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

with open('models/scaler.pkl','rb') as f: scaler=pickle.load(f)
with open('models/logistic_regression.pkl','rb') as f: lr=pickle.load(f)
with open('models/random_forest.pkl','rb') as f: rf=pickle.load(f)
with open('models/xgboost.pkl','rb') as f: xgb=pickle.load(f)

X_test_sc = scaler.transform(X_test)
print('Loaded successfully')

## 3. Predictions

In [ ]:
lr_pred  = lr.predict(X_test_sc)
rf_pred  = rf.predict(X_test)
xgb_pred = xgb.predict(X_test)

lr_prob  = lr.predict_proba(X_test_sc)[:,1]
rf_prob  = rf.predict_proba(X_test)[:,1]
xgb_prob = xgb.predict_proba(X_test)[:,1]

## 4. Classification Reports

In [ ]:
for name, pred in [('Logistic Regression',lr_pred),('Random Forest',rf_pred),('XGBoost',xgb_pred)]:
    print(f'\n=== {name} ===')
    print(classification_report(y_test, pred, target_names=['Non-Fraud','Fraud']))

## 5. Confusion Matrices

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(16,4))
for ax,(name,pred) in zip(axes,[('Logistic Regression',lr_pred),('Random Forest',rf_pred),('XGBoost',xgb_pred)]):
    cm = confusion_matrix(y_test,pred)
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax,
                xticklabels=['Non-Fraud','Fraud'],yticklabels=['Non-Fraud','Fraud'])
    ax.set_title(name)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.suptitle('Confusion Matrices',fontweight='bold')
plt.tight_layout()
plt.show()

## 6. ROC-AUC Curves

In [ ]:
plt.figure(figsize=(8,6))
for name,prob in [('Logistic Regression',lr_prob),('Random Forest',rf_prob),('XGBoost',xgb_prob)]:
    fpr,tpr,_=roc_curve(y_test,prob)
    auc=roc_auc_score(y_test,prob)
    plt.plot(fpr,tpr,label=f'{name} (AUC={auc:.4f})')
plt.plot([0,1],[0,1],'k--',label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-AUC Curve')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Precision-Recall Curves

In [ ]:
plt.figure(figsize=(8,6))
for name,prob in [('Logistic Regression',lr_prob),('Random Forest',rf_prob),('XGBoost',xgb_prob)]:
    prec,rec,_=precision_recall_curve(y_test,prob)
    plt.plot(rec,prec,label=name)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Model Comparison Summary

In [ ]:
rows=[]
for name,pred,prob in [('Logistic Regression',lr_pred,lr_prob),
                        ('Random Forest',rf_pred,rf_prob),
                        ('XGBoost',xgb_pred,xgb_prob)]:
    rows.append({'Model':name,
                 'F1-Score (Fraud)':f1_score(y_test,pred).round(4),
                 'ROC-AUC':roc_auc_score(y_test,prob).round(4)})
comp = pd.DataFrame(rows).set_index('Model')
print(comp)
comp.plot(kind='bar',figsize=(9,4),title='Model Comparison')
plt.tight_layout()
plt.show()